## `Priorities this week:`

#### `[Resolved]` (1/3) **Correct calculations saved and correct plots produced of observed ambient spectra.**

[1]|`Resolved` |Move the pre_filt band out to 500s.
___

[2]|`Resolved` |Remove response is done. Before closing it up:|
> a. Seismic spectra throughout ATaCR should be used only on displacement units, unless used for plots. 

> b. Make sure the conversion to acceleration is only done for the plots.
___

[3]|`Resolved`|Check that 2x thing in the PSD plots.
___

#### `[Incomplete]` (2/3) **The Admittance Puzzle**

[4] |`Incomplete`|Pressure, up to before the TFs script atleast, is fine once I confirm how admittance is calculated (the ZP vs PZ problem seen in plots).
___

[5]|`Incomplete`|Verify that admittance is loaded in the correct form (not the mag-reciprocal, see above) and used correctly in the TF stage.|
___

[6]|`Incomplete`|Confirm each TF is grabbing the right pieces.
___

#### `[Paused]` (3/3) **Misc.**

[7]|`Resolved`|Google doc to organize links to all the notebooks.
___

> Link: https://docs.google.com/document/d/1JkIHQE88BFj2ECcgBMxoShUjtGElEVE88jBPpd8EiuA/edit?usp=sharing
___

[8]|`No time`|One clean notebook for the complete ATaCR method.
___

![Bar](znb_images/Untitled-1.png)
### (1/3) **Correct calculations saved and correct plots produced of observed ambient spectra.**
![Bar](znb_images/Untitled-1.png)


__<a id="section_ID"></a>__
### [`1`]|`Resolved`|`Move the pre_filt band out to 500s.`
>> Done, the pre-filt used in remove response, for both pressure and seismic, is now: [0.001, 0.002, 45.0, 50.0]
___


### [`2`] |`Resolved`|`Remove response is done. Before closing it up:`
>> ##### a. Seismic spectra throughout ATaCR should be used only on displacement units, unless used for plots. 
>> ##### b. Make sure the conversion to acceleration is only done for the plots.

### The log-diff to acceleration now lives exclusively in the plot script ([plotting.py](https://github.com/charleshoots/ATACR_HPS_Comp/blob/main/Packages/ATaCR/OBStools/obstools/atacr/plotting.py)), anything outside of that file is strictly using spectra of displacements.

>> #### See below for ATaCR spectra output plots from the Day-Spectra and Clean-Spectra stages

### Two days of spectra from the Day-Spectra stage:
![DayAvgSpectraCompleted](znb_images/DayAvgSpectraCompleted.png)

### Station averaged spectra from the Clean-Spectra stage:
![DayAvgSpectraCompleted](znb_images/StaAvgSpectraCompleted.png)

> ### I'm pretty happy with the spectra now. I've got no complaints.
___


### [`3`] |`Resolved`| `Check that 2x thing in the PSD plots.`

Decibels are trying to be a logarithmic measure of power, so if you happen to have two power levels then  
$log10(P2/P1)$ is the number of (notional) bels and 10 times that is the number of decibels.

More commonly however you have two amplitudes (where one is the conjugate of the other in the case of an auto-psd), and power normally goes as the square of amplitude. So the number of decibels is  
$$10log10(A2^2/A1^2)=20log10(A2/A1)$$

As the stft in ATaCR outputs in amplitude-density and not power-density, this is the source of the 2x 'thing' to bring the amplitude spectra into power before taking the log.

In Matlab ATaCR, the procedure from time > fft > psd > log-power psd is **identical** with the same 2x scale in the same spot (see noisecal_dailystaspectra.m for reference).
___

![Bar](znb_images/Untitled-1.png)
### (2/3) **The Admittance Puzzle**
![Bar](znb_images/Untitled-1.png)


### [`4`]|`Incomplete`|`Pressure, up to TFs atleast, is fine once I confirm how Admittance is calculated (the ZP vs PZ problem seen in plots).`

> ### Now for the admittance.

___


Column A |
---------|
 A1 | 


Basic observations:

#### In Matlab-ATaCR:

[A] | ML-ATaCRs daily cross-spec for ZP in script: 

[Link](https://github.com/helenjanisz/ATaCR/blob/9ebefec3f2e6ffcb59497df569c67f516ab7e5d8/function/noisecal_dailystaspectra.m#L50) |
---------|
 cpz = spectrum_P(iwin,:)'.*cspectrum_Z(iwin,:)'*2/(NFFT*dt); | 

[`Note`]: cspectrum_Z is the conjugate of the Z spectra, calculated [here (link)](https://github.com/helenjanisz/ATaCR/blob/9ebefec3f2e6ffcb59497df569c67f516ab7e5d8/function/speccal_dailystaspectra.m#L24)  and saved to file for the above operation. 
 [Link](https://github.com/helenjanisz/ATaCR/blob/9ebefec3f2e6ffcb59497df569c67f516ab7e5d8/function/speccal_dailystaspectra.m#L24) |
---------|
 cspec_C = conj(spec_C); | 


[B] | Using the above csd's from the daily-stage for P-to-Z, ML-ATaCR admittance for P-to-Z is calculated here:
 
 [Link](https://github.com/helenjanisz/ATaCR/blob/9ebefec3f2e6ffcb59497df569c67f516ab7e5d8/b2_cleanstaspectra.m#L131) | 
---------|
 ad_stack(:,ie,4) = (abs(specprop.cross.cpz_stack)./specprop.power.cpp_stack); |

Summary of the exact math followed by the above lines of MATLAB code:
$$PZ.Admittance = \frac{|cPZ|}{cPP}  =  \frac{|ftP*Conj(ftZ)|}{ftP*Conj(ftP)}$$

[`Note`]: I am using the same naming convention in Bell 2015 to describe the cpz variable in ML-ATaCR as 'P-to-Z' where this csd describes overlapping the density of the power spectra of the Pressure trace, in pressure trace-spectra units, that is shared with the power spectra of the vertical, in vertical trace-spectra units. As such, 'PZ' admittance (what ML-ATaCR calls it in plots) in the way it is calculated above describes the amplitude gain transfer function that, when multiplied by the PRESSURE trace spectra, shows the same spectra in units of VERTICAL power spectra. For compliance corrections, this TF applied to an event PRESSURE trace predicts the amount of the vertical spectra to remove.

[C] | Admittance is then loaded and plotted [here (link)](https://github.com/helenjanisz/ATaCR/blob/9ebefec3f2e6ffcb59497df569c67f516ab7e5d8/function/plot_cleanstaspectra.m#L88)

[D] From here, the P-to-Z (cpz) csd moves to the TF stage. As of right now, proper comparisons there with Py-ATaCR still need to be done here.

[`Note`]: Most important, the result from the above admittance yields the spectra to subtract from an event vertical trace spectra for compliance correction (using the PZ-TF).

#### In Python ATaCR:

[A] | Py-ATaCR's csd is 'REVERSED' relative to ML-ATaCR.

[Link](https://github.com/nfsi-canada/OBStools/blob/7a08e26887f576c6d9042cdb01b922844423107a/obstools/atacr/classes.py#L757) |
---------|
 cZP = np.mean(self.ftZ[self.goodwins, :] * np.conj(self.ftP[self.goodwins, :]), axis=0) | 

[B] | And normed with the Z-PSD to yield Z-to-P admittance:

[Link](https://github.com/nfsi-canada/OBStools/blob/7a08e26887f576c6d9042cdb01b922844423107a/obstools/scripts/atacr_clean_spectra.py#L487) |
---------|
ad_ZP_all.append(utils.smooth(utils.admittance(daynoise.cross.cZP, daynoise.power.cZZ), 50)) | 

Summary of the exact math followed by the above lines of Python code:
$$ZP.Admittance = \frac{|cZP|}{cZZ}  =  \frac{|Conj(ftP)*ftZ|}{ftZ*Conj(ftZ)}$$


Note: This is at the crux of the possible issue in Py-ATaCR. This csd is the Z-to-P takes the conjugate of P instead of Z (ie how ML-ATaCR does it) and thus has units of PRESSURE spectral density. This describes the amount vertical power spectra mapped onto the pressure power spectra. When used for admittance, we would see the transfer function necessary to transform (multiply) the VERTICAL trace power spectra into PRESSURE trace power spectra. This is the opposite operation of ML-ATaCR. 


> #### At this point, I have stopped as I just don't have the time to spare between now and end of comps. I will most likely find random points in the next week before comps to glance at this. But in terms of any specific time to sit down and completely solve this entire thing for good definitely will not happen until after comps.

[`Final Note`]: What I have shown above is completely accurate but i think it's possible this is not a problem at all if used correctly during TF calcs and event corrections. I need to check the math on this still but you should be able to produce the correct compliance correction using the Z-to-P TF made in Py-ATaCR instead of a P-to-Z TF (as made in ML-ATaCR) by dividing the conjugate by the noise pressure PSD.

As follows from cZP,

$$cZP = |Conj(ftP)*ftZ|$$

ftP,ftZ > 0,

$$Conj(cZP) = |ftP*Conj(ftZ)| = cPZ$$

$$\frac{Conj(cZP)}{cPP} = PZ.Admittance$$

If this is done in the TF stage of PY-ATaCR, then we're good. Just need to find a minute to read it closely and confirm...


![Bar](znb_images/Untitled-1.png)
### (3/3) **Misc.**
![Bar](znb_images/Untitled-1.png)

### [7]|`Resolved`|Google doc to organize links to all the notebooks.
### > Link: https://docs.google.com/document/d/1JkIHQE88BFj2ECcgBMxoShUjtGElEVE88jBPpd8EiuA/edit?usp=sharing
---
### [8]|`No time`|One clean notebook for the complete ATaCR method.